In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================
# PARAMETERS
# =========================
alpha = 1.0

In [ ]:
# NEURAL NETWORK
# input: (x,t)
# output: u
# =========================
class PINN(nn.Module):
    def __init__(self):
        super(PINN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )
        
    def forward(self, x, t):
        inputs = torch.cat([x, t], dim=1)
        return self.net(inputs)

model = PINN().to(device)

In [ ]:
# SAMPLING POINTS
# =========================
N_f = 10000   # collocation points
N_b = 2000    # boundary points
N_i = 2000    # initial points

# domain: x in [0,1], t in [0,1]
x_f = torch.rand((N_f,1), device=device)
t_f = torch.rand((N_f,1), device=device)

# boundary x=0
x_b0 = torch.zeros((N_b//2,1), device=device)
t_b0 = torch.rand((N_b//2,1), device=device)

# boundary x=1
x_b1 = torch.ones((N_b//2,1), device=device)
t_b1 = torch.rand((N_b//2,1), device=device)

# initial t=0
x_i = torch.rand((N_i,1), device=device)
t_i = torch.zeros((N_i,1), device=device)


In [ ]:
# INITIAL CONDITION
# u(x,0) = sin(pi x) + 0.5 sin(3pi x)
# =========================
def u_initial(x):
    return torch.sin(np.pi * x) + 0.5 * torch.sin(3 * np.pi * x)

# =========================

In [ ]:
# LOSS FUNCTION
# =========================
def loss_function():
    
    # --- PDE LOSS ---
    x_f.requires_grad = True
    t_f.requires_grad = True
    
    u = model(x_f, t_f)
    
    u_t = torch.autograd.grad(u, t_f, grad_outputs=torch.ones_like(u),
                             create_graph=True)[0]
    
    u_x = torch.autograd.grad(u, x_f, grad_outputs=torch.ones_like(u),
                             create_graph=True)[0]
    
    u_xx = torch.autograd.grad(u_x, x_f, grad_outputs=torch.ones_like(u_x),
                              create_graph=True)[0]
    
    f = u_t - alpha * u_xx
    loss_pde = torch.mean(f**2)
    
    # --- BOUNDARY LOSS ---
    u_b0 = model(x_b0, t_b0)
    u_b1 = model(x_b1, t_b1)
    
    loss_b = torch.mean((u_b0 - 0)**2) + \
             torch.mean((u_b1 - torch.sin(2*np.pi*t_b1))**2)
    
    # --- INITIAL LOSS ---
    u_i_pred = model(x_i, t_i)
    u_i_true = u_initial(x_i)
    
    loss_i = torch.mean((u_i_pred - u_i_true)**2)
    
    return loss_pde + loss_b + loss_i


In [ ]:
# TRAINING
# =========================
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 5000

for epoch in range(epochs):
    optimizer.zero_grad()
    loss = loss_function()
    loss.backward()
    optimizer.step()
    
    if epoch % 500 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}")


In [ ]:
# PLOTTING RESULTS
# =========================
x = np.linspace(0,1,100)
t_values = [0.0, 0.25, 0.5, 0.75, 1.0]

plt.figure(figsize=(8,5))

for t_val in t_values:
    x_tensor = torch.tensor(x, dtype=torch.float32).unsqueeze(1).to(device)
    t_tensor = torch.full_like(x_tensor, t_val).to(device)
    
    u_pred = model(x_tensor, t_tensor).detach().cpu().numpy()
    
    plt.plot(x, u_pred, label=f"t={t_val}")

plt.legend()
plt.xlabel("x")
plt.ylabel("u(x,t)")
plt.title("PINN Solution")
plt.grid()
plt.show()